# Notebook Overview — Extract Frequency-Domain Features

## Purpose

This notebook extracts the **frequency-domain feature set** from preprocessed images using the train or test metadata CSV generated earlier in the pipeline.

Frequency-domain features capture spectral energy distribution and structural patterns in the frequency space, providing complementary information to gradient and spatial features for distinguishing real images from AI-generated images.

---

## Inputs

The notebook expects the following:

* Metadata CSV files from the GitHub repository:

  * `metadata/splits/train_metadata.csv`
  * `metadata/splits/test_metadata.csv`

* Preprocessed image archive from Google Drive:

  * `/content/drive/MyDrive/DIP_Project/releases/preprocessed/All_Sources_preprocessed.zip`

---

## Assumptions

* Each metadata CSV contains the columns:

  * `filename`
  * `class_label`
  * `source_dataset`
  * `subset`

* The metadata was created previously by the combine-and-split notebook

* The preprocessed images are stored in a single ZIP archive containing all images at the archive root (no subdirectories)

* The notebook extracts images into a local runtime directory:

  * `data/preprocessed/images/`

* Image paths are constructed using filenames from metadata only

* All preprocessed images are:

  * grayscale
  * resized to **256 × 256**

* This notebook does **not** download, split, or preprocess images

* This notebook processes one subset at a time using metadata-driven image selection

* Feature extraction is performed separately for the train and test subsets to preserve split integrity

* All outputs are written to **local runtime storage only**

* No files are written back to Google Drive

---

## What This Notebook Does

### Cell 1 — Environment Setup

* Mount Google Drive to access the preprocessed image archive
* Clone the GitHub repository into the local runtime
* Import project configuration
* Select the target subset (`train` or `test`)
* Verify required metadata and ZIP file inputs

---

### Cell 2 — Extract Preprocessed Images (if needed)

* Check whether images have already been extracted in the local runtime
* If not present, extract:

  * `All_Sources_preprocessed.zip`

  into:

  * `data/preprocessed/images/`

* Verify that extracted PNG images are available

---

### Cell 3 — Define Frequency Feature Functions

* Compute 2D Fourier Transform of images
* Analyze frequency magnitude spectrum
* Extract spectral statistics and radial distributions

---

### Cell 4 — Preview and Validate Sample Image

* Load a sample image using metadata
* Verify image format and dimensions
* Compute and display frequency-domain features
* Visualize magnitude spectrum and radial profile

---

### Cell 5 — Extract Frequency-Domain Features

For each image in the selected subset:

1. Load the image from the extracted image directory
2. Compute frequency-domain representations
3. Extract spectral features
4. Store features together with metadata fields

The frequency feature group includes:

* Low Frequency Energy Ratio
* High Frequency Energy Ratio
* Radial Mean
* Radial Std
* Radial Entropy
* Spectral Centroid
* Spectral Bandwidth
* Log Spectrum Std

---

### Cell 6 — Save Output

* Save extracted features to a subset-specific CSV file
* Verify file creation and row count

---

### Cell 7 — Final Summary

* Display processing summary
* Provide guidance for next step in the pipeline

---

## Outputs

The following files are written to the **local runtime only**:

* `metadata/features/train_frequency_features.csv`
* `metadata/features/test_frequency_features.csv`

These files are not written back to Google Drive and will be lost when the runtime ends unless explicitly preserved.

---

## Expected Sizes

* `train_frequency_features.csv` → **14,400 rows**
* `test_frequency_features.csv` → **3,600 rows**

Each row corresponds to one image and includes metadata plus extracted frequency-domain features.

---

## Notes

* The project contains **6 total sources** and **18,000 images**
* Each source contributes **3,000 images**
* The split is performed earlier with exact per-source counts:

  * **train = 2400**
  * **test = 600**

* This notebook extracts only the **frequency-domain feature group (8 features)**

* Frequency-domain features complement:
  * Gradient features (04A)
  * Spatial features (04B)

* All feature outputs are written under `metadata/features/` in the local runtime

* The preprocessed images are distributed as a single compressed archive for convenience

* This notebook operates entirely from structured metadata and does not determine dataset membership from directory traversal

* This notebook processes one subset per run. To generate both train and test feature files, the notebook must be executed twice.

---

**Next step:**  
Set `SUBSET_NAME` to the desired subset and run the notebook. After both train and test frequency feature files are generated, proceed to **05_Build_Feature_Vectors.ipynb**.

In [ ]:
# ============================================
# Cell 1: Startup (Environment + Verification)
# ============================================

import os
import sys
import zipfile
from pathlib import Path

# -------------------------------------------------
# Mount Google Drive (read input ZIP only)
# -------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

# -------------------------------------------------
# Clone repo into Colab runtime (if not already present)
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_SUBSET,
    TEST_SUBSET,
    TRAIN_METADATA_FILENAME,
    TEST_METADATA_FILENAME,
)

# -------------------------------------------------
# Notebook 04C path overrides
# -------------------------------------------------
METADATA_ROOT = REPO_DIR / "metadata"
SPLITS_DIR = METADATA_ROOT / "splits"
FEATURES_DIR = METADATA_ROOT / "features"

DATA_DIR = REPO_DIR / "data"
PREPROCESSED_DIR = DATA_DIR / "preprocessed"
EXTRACTED_IMAGE_DIR = PREPROCESSED_DIR / "images"

# Input ZIP comes from Google Drive
DRIVE_BASE = Path("/content/drive/MyDrive/DIP_Project")
PREPROCESSED_ZIP = DRIVE_BASE / "releases" / "preprocessed" / "All_Sources_preprocessed.zip"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Select subset to process
# -------------------------------------------------
SUBSET_NAME = TRAIN_SUBSET   # change to TEST_SUBSET when needed

if SUBSET_NAME == TRAIN_SUBSET:
    METADATA_FILE = SPLITS_DIR / TRAIN_METADATA_FILENAME
    OUTPUT_FILE = FEATURES_DIR / "train_frequency_features.csv"
elif SUBSET_NAME == TEST_SUBSET:
    METADATA_FILE = SPLITS_DIR / TEST_METADATA_FILENAME
    OUTPUT_FILE = FEATURES_DIR / "test_frequency_features.csv"
else:
    raise ValueError(f"Invalid SUBSET_NAME: {SUBSET_NAME}")

# -------------------------------------------------
# Verify required inputs
# -------------------------------------------------
print("Verifying required inputs...\n")

if not METADATA_FILE.exists():
    raise FileNotFoundError(f"Missing metadata file: {METADATA_FILE}")

if not PREPROCESSED_ZIP.exists():
    raise FileNotFoundError(f"Missing preprocessed image ZIP file: {PREPROCESSED_ZIP}")

print("Required input files are present.")
print(f"Metadata file:    {METADATA_FILE}")
print(f"Preprocessed ZIP: {PREPROCESSED_ZIP}")
print(f"Extracted images: {EXTRACTED_IMAGE_DIR}")
print(f"Output file:      {OUTPUT_FILE}")

# -------------------------------------------------
# Extract ZIP only if needed
# -------------------------------------------------
existing_png_count = len(list(EXTRACTED_IMAGE_DIR.glob("*.png")))

if existing_png_count > 0:
    print(f"\nExtracted image directory already contains {existing_png_count} PNG files.")
    print("Skipping extraction.")
else:
    print("\nExtracting All_Sources_preprocessed.zip to local runtime ...")
    with zipfile.ZipFile(PREPROCESSED_ZIP, "r") as zip_ref:
        zip_ref.extractall(EXTRACTED_IMAGE_DIR)
    print("Extraction complete.")

# -------------------------------------------------
# Basic verification after extraction
# -------------------------------------------------
image_files = list(EXTRACTED_IMAGE_DIR.glob("*.png"))

if len(image_files) == 0:
    raise FileNotFoundError(
        f"No PNG files found in extracted image directory: {EXTRACTED_IMAGE_DIR}"
    )

print(f"Found {len(image_files)} extracted PNG files.")
print("\nStartup complete.")



In [ ]:
# ============================================
# Cell 2: Load Metadata
# ============================================

import pandas as pd

# -------------------------------------------------
# Load selected subset metadata
# -------------------------------------------------
df = pd.read_csv(METADATA_FILE)

print(f"Loaded metadata for subset: {SUBSET_NAME}")
print(f"Metadata shape: {df.shape}")

# -------------------------------------------------
# Basic column verification
# -------------------------------------------------
required_columns = ["filename", "class_label", "source_dataset", "subset"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Metadata file is missing required columns: {missing_columns}")

print("Required metadata columns verified.")

# -------------------------------------------------
# Verify subset consistency
# -------------------------------------------------
unique_subsets = sorted(df["subset"].dropna().unique().tolist())

if unique_subsets != [SUBSET_NAME]:
    raise ValueError(
        f"Metadata subset mismatch. Expected only '{SUBSET_NAME}', found: {unique_subsets}"
    )

print(f"Subset column verified: {unique_subsets}")

# -------------------------------------------------
# Build image paths from filenames
# -------------------------------------------------
df["image_path"] = df["filename"].apply(lambda x: EXTRACTED_IMAGE_DIR / x)

# -------------------------------------------------
# Verify image files exist
# -------------------------------------------------
missing_images = df.loc[~df["image_path"].apply(lambda p: p.exists()), "filename"].tolist()

if missing_images:
    raise FileNotFoundError(
        "Some image files referenced by metadata were not found in the extracted image directory.\n"
        f"First missing files: {missing_images[:10]}"
    )

print("All metadata-referenced image files were found.")

# -------------------------------------------------
# Display summary
# -------------------------------------------------
print("\nClass distribution:")
print(df["class_label"].value_counts())

print("\nSource distribution:")
print(df["source_dataset"].value_counts())

print("\nSample rows:")
display(df.head())



In [ ]:
# ============================================
# Cell 3: Frequency Feature Helper Functions
# ============================================

import numpy as np
from scipy.stats import entropy

# -------------------------------------------------
# Compute centered 2D FFT
# -------------------------------------------------
def compute_fft(img):
    F = np.fft.fft2(img)
    F_shift = np.fft.fftshift(F)
    return F_shift

# -------------------------------------------------
# Magnitude / log-magnitude / power spectrum
# -------------------------------------------------
def compute_magnitude_spectrum(F_shift):
    mag = np.abs(F_shift)
    log_mag = np.log1p(mag)
    power = mag ** 2
    return mag, log_mag, power

# -------------------------------------------------
# Radial mean power profile
# -------------------------------------------------
def radial_profile(power_spectrum):
    h, w = power_spectrum.shape
    cy, cx = h // 2, w // 2

    y, x = np.indices((h, w))
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    r = r.astype(np.int32)

    max_r = r.max()
    radial_mean = np.zeros(max_r + 1, dtype=np.float32)

    for i in range(max_r + 1):
        mask = (r == i)
        if np.any(mask):
            radial_mean[i] = power_spectrum[mask].mean()

    return radial_mean

# -------------------------------------------------
# Entropy of radial profile
# -------------------------------------------------
def radial_entropy(radial_vals):
    hist, _ = np.histogram(radial_vals, bins=64)
    hist = hist.astype(np.float64)
    hist = hist / (hist.sum() + 1e-12)
    hist = np.clip(hist, 1e-12, None)
    return float(entropy(hist, base=2))

# -------------------------------------------------
# Frequency band energy ratios
# -------------------------------------------------
def frequency_band_ratios(power_spectrum):
    h, w = power_spectrum.shape
    cy, cx = h // 2, w // 2

    y, x = np.indices((h, w))
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)

    max_r = np.max(r)

    low_mask = r <= max_r * 0.2
    mid_mask = (r > max_r * 0.2) & (r <= max_r * 0.5)
    high_mask = r > max_r * 0.5

    total_energy = power_spectrum.sum() + 1e-12

    low_ratio = float(power_spectrum[low_mask].sum() / total_energy)
    mid_ratio = float(power_spectrum[mid_mask].sum() / total_energy)
    high_ratio = float(power_spectrum[high_mask].sum() / total_energy)

    return low_ratio, mid_ratio, high_ratio

# -------------------------------------------------
# Spectral centroid and bandwidth
# -------------------------------------------------
def spectral_centroid(radial_vals):
    indices = np.arange(len(radial_vals))
    total = radial_vals.sum() + 1e-12
    centroid = float((indices * radial_vals).sum() / total)
    return centroid

def spectral_bandwidth(radial_vals, centroid):
    indices = np.arange(len(radial_vals))
    total = radial_vals.sum() + 1e-12
    variance = ((indices - centroid) ** 2 * radial_vals).sum() / total
    return float(np.sqrt(variance))

# -------------------------------------------------
# Extract frequency-domain features
# -------------------------------------------------
def extract_frequency_features(img):
    F_shift = compute_fft(img)
    mag, log_mag, power = compute_magnitude_spectrum(F_shift)

    radial_vals = radial_profile(power)

    low_ratio, mid_ratio, high_ratio = frequency_band_ratios(power)

    r_mean = float(np.mean(radial_vals))
    r_std = float(np.std(radial_vals))
    r_entropy = radial_entropy(radial_vals)

    centroid = spectral_centroid(radial_vals)
    bandwidth = spectral_bandwidth(radial_vals, centroid)

    log_std = float(np.std(log_mag))

    features = {
        "Low Frequency Energy Ratio": low_ratio,
        "High Frequency Energy Ratio": high_ratio,
        "Radial Mean": r_mean,
        "Radial Std": r_std,
        "Radial Entropy": r_entropy,
        "Spectral Centroid": centroid,
        "Spectral Bandwidth": bandwidth,
        "Log Spectrum Std": log_std,
    }

    return features, log_mag, power, radial_vals

print("Frequency helper functions defined.")



In [ ]:
# ============================================
# Cell 4: Preview and Validate Sample Image
# ============================================

import cv2
import matplotlib.pyplot as plt

# -------------------------------------------------
# Select a sample image from metadata
# -------------------------------------------------
sample_row = df.iloc[0]
sample_path = sample_row["image_path"]

print("Sample metadata row:")
display(sample_row)

print(f"\nSample image path: {sample_path}")

if not sample_path.exists():
    raise FileNotFoundError(f"Sample image not found: {sample_path}")

# -------------------------------------------------
# Load sample image in grayscale
# -------------------------------------------------
sample_img = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)

if sample_img is None:
    raise ValueError(f"Could not load sample image: {sample_path}")

print(f"Loaded sample image shape: {sample_img.shape}")
print(f"Loaded sample image dtype: {sample_img.dtype}")

# -------------------------------------------------
# Verify expected image properties
# -------------------------------------------------
if len(sample_img.shape) != 2:
    raise ValueError("Expected grayscale image with 2 dimensions.")

if sample_img.shape != (256, 256):
    raise ValueError(f"Expected image shape (256, 256), found {sample_img.shape}")

print("Sample image format verified.")

# -------------------------------------------------
# Compute frequency features for sample image
# -------------------------------------------------
sample_features, log_mag, power, radial_vals = extract_frequency_features(sample_img)

print("\nSample frequency-domain features:")
for key, value in sample_features.items():
    print(f"{key}: {value:.6f}")

print(f"\nNumber of features extracted: {len(sample_features)}")

# -------------------------------------------------
# Display sample image and derived frequency views
# -------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(sample_img, cmap="gray")
axes[0].set_title("Sample Image")
axes[0].axis("off")

axes[1].imshow(log_mag, cmap="gray")
axes[1].set_title("Log Magnitude Spectrum")
axes[1].axis("off")

axes[2].imshow(np.log1p(power), cmap="gray")
axes[2].set_title("Log Power Spectrum")
axes[2].axis("off")

axes[3].plot(radial_vals)
axes[3].set_title("Radial Profile")
axes[3].set_xlabel("Radius")
axes[3].set_ylabel("Mean Power")

plt.tight_layout()
plt.show()



In [ ]:
# ============================================
# Cell 5: Extract Frequency-Domain Features for Subset
# ============================================

from tqdm.notebook import tqdm

# -------------------------------------------------
# Extract features for all images in selected subset
# -------------------------------------------------
records = []

print(f"Beginning frequency-domain feature extraction for subset: {SUBSET_NAME}")
print(f"Total images to process: {len(df)}\n")

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {SUBSET_NAME} images"):
    image_path = row["image_path"]

    try:
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError(f"Could not load image: {image_path}")

        features, _, _, _ = extract_frequency_features(img)

        record = {
            "filename": row["filename"],
            "class_label": row["class_label"],
            "source_dataset": row["source_dataset"],
            "subset": row["subset"],
        }
        record.update(features)
        records.append(record)

    except Exception as e:
        print(f"Skipping {row['filename']} due to error: {e}")

# -------------------------------------------------
# Convert to dataframe
# -------------------------------------------------
frequency_features_df = pd.DataFrame(records)

# -------------------------------------------------
# Verify output structure
# -------------------------------------------------
expected_columns = [
    "filename",
    "class_label",
    "source_dataset",
    "subset",
    "Low Frequency Energy Ratio",
    "High Frequency Energy Ratio",
    "Radial Mean",
    "Radial Std",
    "Radial Entropy",
    "Spectral Centroid",
    "Spectral Bandwidth",
    "Log Spectrum Std",
]

missing_columns = [col for col in expected_columns if col not in frequency_features_df.columns]
if missing_columns:
    raise ValueError(f"Missing expected output columns: {missing_columns}")

frequency_features_df = frequency_features_df[expected_columns]

print("\nFrequency-domain feature extraction complete.")
print(f"Output shape: {frequency_features_df.shape}")
print(f"Processed images: {len(frequency_features_df)}")
print(f"Expected images:  {len(df)}")

if len(frequency_features_df) != len(df):
    print("\nWARNING: Some images were skipped during processing.")

print("\nSample output rows:")
display(frequency_features_df.head())



In [ ]:
# ============================================
# Cell 6: Save Output
# ============================================

# -------------------------------------------------
# Save frequency feature dataframe
# -------------------------------------------------
frequency_features_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved frequency features to: {OUTPUT_FILE}")

# -------------------------------------------------
# Verify output file exists
# -------------------------------------------------
if not OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Output file was not created: {OUTPUT_FILE}")

# -------------------------------------------------
# Verify saved row count
# -------------------------------------------------
saved_df = pd.read_csv(OUTPUT_FILE)

if len(saved_df) != len(frequency_features_df):
    raise ValueError(
        f"Saved row count mismatch. Expected {len(frequency_features_df)}, found {len(saved_df)}"
    )

print(f"Verified saved file row count: {len(saved_df)}")
print("\nCell complete.")



In [ ]:
# ============================================
# Cell 7: Final Summary and Next Step
# ============================================

from IPython.display import display, HTML

print("Frequency-domain feature extraction completed successfully.\n")

print(f"Subset processed : {SUBSET_NAME}")
print(f"Input rows       : {len(df)}")
print(f"Output rows      : {len(frequency_features_df)}")
print(f"Output columns   : {len(frequency_features_df.columns)}")
print(f"Saved file       : {OUTPUT_FILE}")

print("\nFeature columns:")
for col in frequency_features_df.columns[4:]:
    print(f"  - {col}")

# -------------------------------------------------
# Next step message (based on subset just run)
# -------------------------------------------------
if SUBSET_NAME == TRAIN_SUBSET:
    message = """
    <b>Next Step:</b><br>
    Set <code>SUBSET_NAME = TEST_SUBSET</code> and rerun this notebook to generate <b>test</b> frequency-domain features.
    """
    border_color = "#ff9800"
    bg_color = "#fff3e0"
else:
    message = f"""
    <b>Current Run Complete:</b><br>
    This run generated <b>{OUTPUT_FILE.name}</b> for the <b>test</b> subset.<br><br>
    After both train and test frequency feature files are available, proceed to
    <b>05_Build_Feature_Vectors.ipynb</b>.
    """
    border_color = "#4CAF50"
    bg_color = "#E8F5E9"

display(HTML(f"""
<div style="
    padding: 15px;
    border: 2px solid {border_color};
    background-color: {bg_color};
    border-radius: 8px;
    font-size: 16px;
">
{message}
</div>
"""))

